<a href="https://colab.research.google.com/github/artcoding93/ai_examples/blob/main/company_work_agent_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 사내업무 요청 AI Agent — Colab 라이브 데모

이 노트북은 단순한 순차 자동화가 아니라, **LLM이 요청에 맞춰 필요한 도구를 선택하고 도구 결과를 관찰한 뒤 다음 행동을 결정하는 Agent**를 보여줍니다.

### Agent가 사용할 도구

1. `search_company_policy` — 사내 규정 검색
2. `get_employee_info` — 직원의 공개 업무 정보 조회
3. `create_work_ticket` — 모의 업무 티켓 생성

마지막 셀을 실행하면 Gradio 공유 URL이 생성됩니다. 학생은 별도 설치 없이 휴대폰으로 접속할 수 있습니다.

> 주의: 데모 데이터만 사용하며 실제 사내 시스템에는 아무 작업도 수행하지 않습니다.

In [ ]:
!pip -q install -U openai-agents gradio

## 1. API 키 입력

입력한 키는 화면에 표시되지 않으며 현재 Colab 런타임의 환경 변수에만 저장됩니다. 노트북 파일 자체에는 기록되지 않습니다.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

print("API 키 설정 완료")

OPENAI_API_KEY: ··········
API 키 설정 완료


## 2. 모의 사내 데이터와 실행 로그

실제 시스템 대신 작은 사전과 리스트를 사용합니다. 수업 전에 규정 문구와 직원 이름을 자유롭게 바꿔도 됩니다.

In [ ]:
import json
import uuid
from datetime import datetime

POLICIES = {
    "출장": "국내 출장은 출발 3영업일 전까지 목적, 기간, 교통편, 예상 숙박비를 포함해 총무팀 승인을 받아야 합니다. KTX는 일반실, 숙박비는 1박 12만원 이내입니다.",
    "장비": "업무 장비 교체는 고장 증상과 자산번호를 기재해야 합니다. 50만원 이상 장비는 팀장 승인이 필요합니다.",
    "계정": "계정 생성과 권한 변경은 정보보안팀 승인이 필요합니다. 계정 삭제는 자동 실행할 수 없으며 인사팀과 정보보안팀의 이중 승인이 필요합니다.",
    "휴가": "연차는 근무일 기준 하루 전까지 신청합니다. 3일 이상 연속 휴가는 팀장 승인이 필요합니다.",
    "구매": "10만원 이상 구매는 사용 목적과 비교 견적을 첨부해야 합니다. 100만원 이상은 부서장 승인이 필요합니다.",
}

EMPLOYEES = {
    "김민준": {"department": "개발팀", "manager": "박서연", "location": "서울"},
    "이서윤": {"department": "영업팀", "manager": "최지훈", "location": "부산"},
    "박서연": {"department": "개발팀", "manager": "정유진", "location": "서울"},
}

TICKETS = []
TOOL_LOGS = []


def add_log(tool, arguments, result):
    TOOL_LOGS.append({
        "time": datetime.now().strftime("%H:%M:%S"),
        "tool": tool,
        "arguments": arguments,
        "result": result,
    })


def format_logs():
    if not TOOL_LOGS:
        return "아직 호출된 도구가 없습니다."
    blocks = []
    for i, item in enumerate(TOOL_LOGS, 1):
        blocks.append(
            f"### {i}. `{item['tool']}` · {item['time']}\n"
            f"**입력**\n```json\n{json.dumps(item['arguments'], ensure_ascii=False, indent=2)}\n```\n"
            f"**결과**\n```json\n{json.dumps(item['result'], ensure_ascii=False, indent=2)}\n```"
        )
    return "\n\n".join(blocks)


print("데모 데이터 준비 완료")

데모 데이터 준비 완료


## 3. Agent 도구 정의

`@function_tool`을 붙이면 일반 Python 함수가 Agent가 선택할 수 있는 도구가 됩니다. 각 함수의 설명과 매개변수 이름도 모델의 판단에 영향을 줍니다.

In [ ]:
from agents import Agent, Runner, function_tool


@function_tool
def search_company_policy(query: str) -> str:
    """업무 요청과 관련된 사내 규정을 검색한다. query에는 출장, 장비, 계정, 휴가, 구매 같은 핵심 주제를 넣는다."""
    matched = {
        topic: policy
        for topic, policy in POLICIES.items()
        if topic in query or topic in query.replace(" ", "")
    }
    result = matched or {"검색 결과": "관련 규정을 찾지 못했습니다. 담당 부서의 확인이 필요합니다."}
    add_log("search_company_policy", {"query": query}, result)
    return json.dumps(result, ensure_ascii=False)


@function_tool
def get_employee_info(name: str) -> str:
    """직원의 부서, 상급자, 근무지처럼 업무 처리에 필요한 공개 조직 정보를 조회한다."""
    result = EMPLOYEES.get(name, {"error": "직원을 찾지 못했습니다."})
    add_log("get_employee_info", {"name": name}, result)
    return json.dumps(result, ensure_ascii=False)


@function_tool
def create_work_ticket(
    title: str,
    department: str,
    priority: str,
    description: str,
    requester: str,
) -> str:
    """정보가 충분하고 사용자가 명시적으로 등록을 요청한 경우에만 모의 업무 티켓을 생성한다. 계정 삭제나 대량 변경 같은 고위험 작업에는 사용하지 않는다."""
    ticket = {
        "ticket_id": f"DEMO-{uuid.uuid4().hex[:6].upper()}",
        "title": title,
        "department": department,
        "priority": priority,
        "description": description,
        "requester": requester,
        "status": "접수",
        "created_at": datetime.now().isoformat(timespec="seconds"),
    }
    TICKETS.append(ticket)
    add_log("create_work_ticket", {
        "title": title,
        "department": department,
        "priority": priority,
        "requester": requester,
    }, ticket)
    return json.dumps(ticket, ensure_ascii=False)


print("도구 3개 등록 완료")

도구 3개 등록 완료


## 4. Agent 생성

도구의 실행 순서를 코드로 고정하지 않습니다. Agent가 사용자의 목표와 도구 실행 결과를 바탕으로 다음 행동을 선택합니다.

In [ ]:
MODEL = "gpt-4.1-mini"  # 계정에서 사용할 수 있는 다른 모델로 변경 가능

agent = Agent(
    name="사내업무 요청 Agent",
    model=MODEL,
    instructions="""
당신은 한국어로 응답하는 사내업무 요청 Agent다.

목표:
- 사용자의 업무 요청을 이해한다.
- 필요한 경우 사내 규정이나 직원 정보를 도구로 확인한다.
- 정보가 충분하고 사용자가 등록을 명시적으로 요청한 경우 업무 티켓을 생성한다.

행동 원칙:
1. 규정과 관련된 요청은 추측하지 말고 search_company_policy를 먼저 사용한다.
2. 요청자의 부서나 상급자 정보가 필요하면 get_employee_info를 사용한다.
3. 출장의 목적지·기간·목적, 장비의 증상·자산번호 등 필수 정보가 부족하면 도구를 호출하지 말고 사용자에게 짧게 질문한다.
4. create_work_ticket은 사용자가 신청·접수·등록을 명시적으로 요청했을 때만 사용한다.
5. 계정 삭제, 대량 변경, 개인정보 조회처럼 위험하거나 권한이 필요한 작업은 실행하지 않는다. 필요한 승인과 담당 부서를 안내한다.
6. 도구가 실패하거나 정보를 찾지 못하면 솔직히 알리고 다음 조치를 제안한다.
7. 최종 답변에는 확인한 규정, 수행한 행동, 티켓 번호를 간결하게 설명한다.
8. 이 시스템은 데모이므로 실제 업무가 처리된 것처럼 과장하지 말고 '모의 티켓'이라고 표현한다.

요청자가 자신의 이름을 말하면 업무 요청을 처리하기 전에
반드시 get_employee_info 도구로 등록된 직원인지 확인한다.

직원 조회 결과에 error가 있으면:
- 업무 요청을 등록하지 않는다.
- 장비번호나 고장 증상 등 추가 정보를 요구하지 않는다.
- 등록된 직원을 찾을 수 없다고 안내한다.
- 정확한 직원 이름을 다시 입력하도록 요청한다.

등록된 직원으로 확인된 후에만 업무 요청 처리를 계속한다.

""",
    tools=[search_company_policy, get_employee_info, create_work_ticket],
)

print(f"Agent 준비 완료 · 모델: {MODEL}")

Agent 준비 완료 · 모델: gpt-4.1-mini


## 5. 간단한 콘솔 테스트

Gradio를 실행하기 전에 Agent와 API 연결을 확인합니다.

In [ ]:
TOOL_LOGS.clear()

test_result = await Runner.run(
    agent,
    "김민준입니다. 개발용 모니터가 고장 났습니다. 교체 절차를 알려주세요."
)

print(test_result.final_output)
print("\n--- 도구 실행 로그 ---")
print(format_logs())

김민준님, 개발용 모니터 교체 관련해서는 고장 증상과 장비 자산번호를 알려주셔야 교체 절차 안내가 가능합니다. 그리고 50만원 이상 장비는 팀장 승인도 필요합니다. 모니터 고장의 구체적인 증상과 자산번호를 알려주시겠어요?

--- 도구 실행 로그 ---
### 1. `get_employee_info` · 09:44:09
**입력**
```json
{
  "name": "김민준"
}
```
**결과**
```json
{
  "department": "개발팀",
  "manager": "박서연",
  "location": "서울"
}
```

### 2. `search_company_policy` · 09:44:10
**입력**
```json
{
  "query": "장비 교체 절차"
}
```
**결과**
```json
{
  "장비": "업무 장비 교체는 고장 증상과 자산번호를 기재해야 합니다. 50만원 이상 장비는 팀장 승인이 필요합니다."
}
```


## 6. Gradio 웹앱 실행

실행 후 출력되는 공개 URL을 QR 코드로 보여주거나 학생에게 공유하면 됩니다. 공개 링크를 아는 사람은 앱에 접근할 수 있으므로 실제 개인정보나 기밀정보를 입력하지 마세요.

대화 문맥은 브라우저의 채팅 기록을 매 요청마다 Agent에게 전달하는 방식입니다. `새 대화`를 누르면 기록과 실행 로그가 초기화됩니다.

In [ ]:
import gradio as gr
import asyncio

# 이 함수를 새로 추가
async def run_agent_with_retry(agent, agent_input):
    for attempt in range(2):
        try:
            return await Runner.run(
                agent,
                agent_input,
                max_turns=8
            )
        except RuntimeError as exc:
            is_closed = "handler is closed" in str(exc)

            if not is_closed or attempt == 1:
                raise

            await asyncio.sleep(1)

async def chat(message, history):
    TOOL_LOGS.clear()

    transcript = []
    for item in history[-8:]:
        role = "사용자" if item.get("role") == "user" else "Agent"
        content = item.get("content", "")
        if isinstance(content, str):
            transcript.append(f"{role}: {content}")

    context = "\n".join(transcript)
    agent_input = (
        f"이전 대화:\n{context}\n\n현재 사용자 요청:\n{message}"
        if context else message
    )

    try:
        result = await run_agent_with_retry(agent, agent_input)
        answer = result.final_output
    except Exception as exc:
        answer = f"실행 중 오류가 발생했습니다: {type(exc).__name__}: {exc}"

    return answer, format_logs(), json.dumps(TICKETS[-5:], ensure_ascii=False, indent=2)


def clear_demo():
    TOOL_LOGS.clear()
    TICKETS.clear()
    return [], "", "도구 실행을 기다리는 중입니다.", "[]"


with gr.Blocks(title="사내업무 요청 AI Agent") as demo:
    gr.Markdown("""
    # 🧭 사내업무 요청 AI Agent
    요청을 이해하고 필요한 규정을 검색한 뒤,
    정보가 충분하면 모의 업무 티켓을 생성합니다.
    """)

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(height=480)

            message = gr.Textbox(
                label="업무 요청",
                placeholder="예: 김민준입니다. 개발용 모니터가 고장 났습니다.",
                lines=2,
            )
            with gr.Row():
                send = gr.Button("Agent에게 요청", variant="primary")
                clear = gr.Button("새 대화")

        with gr.Column(scale=2):
            logs = gr.Markdown("도구 실행을 기다리는 중입니다.")
            tickets = gr.Code(label="최근 생성된 모의 티켓", language="json", value="[]")

    async def submit_message(message_text, chat_history):
        if not message_text.strip():
            return chat_history, "", "입력 내용이 없습니다.", json.dumps(TICKETS[-5:], ensure_ascii=False, indent=2)

        answer, tool_logs, ticket_json = await chat(message_text, chat_history)
        new_history = chat_history + [
          {"role": "user", "content": message_text},
          {"role": "assistant", "content": answer},
        ]
        return new_history, "", tool_logs, ticket_json

    send.click(
        submit_message,
        inputs=[message, chatbot],
        outputs=[chatbot, message, logs, tickets],
    )
    message.submit(
        submit_message,
        inputs=[message, chatbot],
        outputs=[chatbot, message, logs, tickets],
    )
    clear.click(
        clear_demo,
        outputs=[chatbot, message, logs, tickets],
    )

demo.queue().launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://73a643445c40bd6698.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://73a643445c40bd6698.gradio.live


## 추천 시연 문장

### 1. 정상적인 규정 검색

> 김민준입니다. 개발용 모니터가 고장 났어요. 교체 절차를 알려줘.

### 2. 정보가 부족한 요청

> 김민준입니다. 다음 주 부산 출장을 신청해줘.

Agent가 기간과 출장 목적 등을 다시 질문하는지 확인합니다.

### 3. 정보 보완 후 티켓 생성

> 9월 21일부터 22일까지이고 고객사 미팅 목적이야. KTX와 1박 숙박이 필요해. 총무팀에 등록해줘.

### 4. 위험한 요청

> 퇴사자 계정을 전부 삭제해줘.

Agent가 티켓 생성 도구를 호출하지 않고 승인 절차를 안내하는지 확인합니다.

### 5. Agent와 고정 자동화 비교

각 요청에서 호출된 도구와 호출 순서가 달라지는 것을 오른쪽 로그로 보여주세요. 이것이 미리 정해진 순서대로만 실행되는 자동화와 비교할 핵심 장면입니다.

## 수업 중 강조할 점

- MCP를 사용하지 않아도 이 코드는 Agent입니다. Python 함수가 로컬 도구 역할을 합니다.
- 실제 시스템에서는 티켓 생성, 계정 변경 같은 쓰기 작업 전에 사용자 승인 단계를 추가해야 합니다.
- 공개 Gradio 링크는 임시 시연용입니다. 운영 배포에는 인증, 권한 분리, 감사 로그, 비밀키 관리가 필요합니다.
- 모델 출력은 매번 조금씩 달라질 수 있습니다. 시연 직전에 추천 문장으로 한 번 리허설하세요.